In [ ]:
%load_ext autoreload
%autoreload 1

In [ ]:
import sys
import os
from logging import debug, info, warning
from warnings import warn

import astropy.units as u
import bokeh
import bokeh.io
import bokeh.models
import bokeh.plotting
import colorcet as cc
import healpy as hp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import panel as pn
from astropy.time import Time, TimeDelta
from astropy.coordinates import AltAz, Angle, EarthLocation, HADec, ICRS, SkyCoord, get_body
from lsst.resources import ResourcePath

from rubin_sim.data import get_baseline
from schedview.collect.visits import read_visits, read_ddf_visits, NIGHT_STACKERS, DDF_STACKERS
from rubin_scheduler.site_models import Almanac

In [ ]:
bokeh.io.output_notebook()
pn.extension

In [ ]:
baseline = "/sdf/group/rubin/web_data/sim-data/sims_featureScheduler_runs5.3/baseline/baseline_v5.3.0_10yrs.db"

In [ ]:
visits = read_visits(20380101, baseline, stackers=DDF_STACKERS, num_nights=20*365)

In [ ]:
visits['tau'] = visits['t_eff'] / visits['visitExposureTime']

In [ ]:
visits.tau.hist(bins=100)

In [ ]:
bands = list('ugrizy')
band_tau = [visits.loc[visits.band == b, 'tau'] for b in bands]

In [ ]:
plt.hist(band_tau, stacked=True, bins=np.arange(0, 2, 0.01), label=bands)
plt.xlabel("effective time / exposure time")
plt.ylabel("# visits")
plt.legend()
plt.show()

In [ ]:
visits.columns

In [ ]:
moon_bin_cuts = np.arange(-1, 90, 1)
num_moon_bins = len(moon_bin_cuts) + 1
visits['moon_bin'] = pd.cut(visits['moonPhase'], bins=moon_bin_cuts)
#visits.loc[visits.moonAlt < 0, 'moon_bin'] = visits.moon_bin.min()

sorted_moon_bins = visits['moon_bin'].cat.categories
visits.loc[visits.moonAlt < 0, 'moon_bin'] = sorted_moon_bins[0]

moon_groups = [visits.loc[visits.moon_bin == mb, 'tau'].values for mb in sorted_moon_bins]
labels = [str(label) for label in sorted_moon_bins]
labels[0] = 'Down'

# Pick colors from the viridis colormap
cmap = mpl.cm.get_cmap('cividis', len(moon_groups))
colors = [cmap(i) for i in range(len(moon_groups))]
colors[0] = '#000000'

plt.hist(
    moon_groups,
    bins=np.arange(0, 2, 0.01),
    stacked=True,
    label=labels,
    color=colors,
)

plt.xlabel('effective time / exposure time')
plt.ylabel('# visits')
plt.title('Histogram of effective time / exposure time colored by moon phase')
plt.tight_layout()
plt.show()


In [ ]:
visits.groupby(visits.moon_bin==sorted_moon_bins[0])['tau'].describe().T

In [ ]:
plt.hist2d(visits.airmass, visits.tau, range=[[1, 3], [0, 2]], bins=100, cmap='bone_r')
plt.xlabel('airmass')
plt.ylabel('effective time / exposure time')

In [ ]:
plt.hist2d(visits.seeingFwhm500, visits.tau, range=[[0.2, 3], [0, 2]], bins=100, cmap='bone_r')
plt.xlabel('seeingFwhm500')
plt.ylabel('effective time / exposure time')

In [ ]:
plt.hist2d(visits.moonDistance, visits.tau, range=[[29, 180], [0, 2]], bins=100, cmap='bone_r')
plt.xlabel('moonDistance')
plt.ylabel('effective time / exposure time')

In [ ]:
zenith_dark_sb = pd.Series({'u': 23.05, 'g': 22.25, 'r': 21.2, 'i': 20.46, 'z': 19.61, 'y': 18.6})

In [ ]:
darkest_sky_in_band = visits.groupby('band')['skyBrightness'].max()
pd.DataFrame({'smtn002': zenith_dark_sb, 'darkest_sky_in_band': darkest_sky_in_band})

In [ ]:
visit_sky = visits.reset_index().set_index('band').loc[:, ['observationId', 'skyBrightness']]
visit_sky['sky_diff'] = (zenith_dark_sb - visit_sky['skyBrightness']).values
visits['sky_diff'] = visit_sky.set_index('observationId').sky_diff

In [ ]:
which_visits = visits.moonAlt < 0
plt.hist2d(visits.loc[which_visits, 'sky_diff'], visits.loc[which_visits, 'tau'], range=[[-0.25, 2], [0, 2]], bins=100, cmap='bone_r')
plt.xlabel('Relative sky brightness')
plt.ylabel('effective time / exposure time')

In [ ]:
sky_mean_tau = 2.512**(-1*visits.sky_diff.mean())
sky_median_tau = 2.512**(-1*visits.sky_diff.median())
print(f"Mean tau from sky: {sky_mean_tau:.2f}, Median tau from sky: {sky_median_tau:0.2f}")

In [ ]:
moon_bin_cuts = np.arange(-1, 90, 1)
num_moon_bins = len(moon_bin_cuts) + 1
visits['moon_bin'] = pd.cut(visits['moonPhase'], bins=moon_bin_cuts)

sorted_moon_bins = visits['moon_bin'].cat.categories

moon_groups = [visits.loc[visits.moon_bin == mb, 'sky_diff'].values for mb in sorted_moon_bins]
labels = [str(label) for label in sorted_moon_bins]

# Pick colors from the viridis colormap
cmap = mpl.cm.get_cmap('cividis', len(moon_groups))
colors = [cmap(i) for i in range(len(moon_groups))]
colors[0] = '#000000'

height = 4
width = height * 1.618
plt.figure(figsize=(width, height))
plt.hist(
    moon_groups,
    bins=np.arange(-0.2, 3, 0.05),
    stacked=True,
    color=colors,
)

plt.axvline(visits.sky_diff.mean(), color='red', label=f'Mean = {visits.sky_diff.mean():.2f}')
plt.axvline(visits.sky_diff.median(), color='red', linestyle='--', label=f'Median = {visits.sky_diff.median():.2f}')

plt.xlabel('SMTN-002 sky brightness - baseline opsim visit skyBrightness')
plt.ylabel('# visits')
plt.title('SMTN-002 sky brightness - baseline opsim visit skyBrightness')
plt.tight_layout()
plt.legend()
plt.show()

# Group by night

In [ ]:
nightly = visits.groupby('day_obs_iso8601').agg({'observationStartMJD': 'median', 'moonPhase': 'median', 'visitExposureTime': 'sum', 't_eff': 'sum'})

In [ ]:
nightly['t_eff_exptime_ratio'] = nightly.t_eff / nightly.visitExposureTime

In [ ]:
sunset_info = pd.DataFrame(Almanac().get_sunset_info(nightly.observationStartMJD))
nightly['sun_n12_setting'] = sunset_info['sun_n12_setting'].values
nightly['sun_n12_rising'] = sunset_info['sun_n12_rising'].values
assert np.all(nightly.observationStartMJD > nightly.sun_n12_setting)
assert np.all(nightly.observationStartMJD < nightly.sun_n12_rising)

nightly['night_length'] = (nightly.sun_n12_rising - nightly.sun_n12_setting)*24*60*60
nightly['t_eff_night_ratio'] = nightly.t_eff / nightly.night_length

In [ ]:
nightly.head()

In [ ]:
t_eff_stats = nightly.loc[:, ['t_eff_exptime_ratio', 't_eff_night_ratio']].describe()
t_eff_stats

In [ ]:
moon_bin_cuts = np.arange(-1, 90, 1)
num_moon_bins = len(moon_bin_cuts) + 1
nightly['moon_bin'] = pd.cut(nightly['moonPhase'], bins=moon_bin_cuts)

sorted_moon_bins = visits['moon_bin'].cat.categories

moon_groups = [nightly.loc[nightly.moon_bin == mb, 't_eff_exptime_ratio'].values for mb in sorted_moon_bins]
labels = [str(label) for label in sorted_moon_bins]

# Pick colors from the viridis colormap
cmap = mpl.cm.get_cmap('cividis', len(moon_groups))
colors = [cmap(i) for i in range(len(moon_groups))]
colors[0] = '#000000'

height = 4
width = height * 1.618
plt.figure(figsize=(width, height))
plt.hist(
    moon_groups,
    bins=np.arange(0, 1.5, 0.05),
    stacked=True,
    color=colors,
)

plt.axvline(t_eff_stats.loc['mean', 't_eff_exptime_ratio'], color='red', label=f'Mean = {t_eff_stats.loc['mean', 't_eff_exptime_ratio']:.2f}')
plt.axvline(t_eff_stats.loc['50%', 't_eff_exptime_ratio'], color='red', linestyle='--', label=f'Median = {t_eff_stats.loc['50%', 't_eff_exptime_ratio']:.2f}')


plt.xlabel('total effective time / total exposure time')
plt.ylabel('# nights')
plt.title('Nightly total effective time / exposure time colored by moon phase')
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
moon_bin_cuts = np.arange(-1, 90, 1)
num_moon_bins = len(moon_bin_cuts) + 1
nightly['moon_bin'] = pd.cut(nightly['moonPhase'], bins=moon_bin_cuts)

sorted_moon_bins = visits['moon_bin'].cat.categories

moon_groups = [nightly.loc[nightly.moon_bin == mb, 't_eff_night_ratio'].values for mb in sorted_moon_bins]
labels = [str(label) for label in sorted_moon_bins]

# Pick colors from the viridis colormap
cmap = mpl.cm.get_cmap('cividis', len(moon_groups))
colors = [cmap(i) for i in range(len(moon_groups))]
colors[0] = '#000000'

height = 4
width = height * 1.618
plt.figure(figsize=(width, height))
plt.hist(
    moon_groups,
    bins=np.arange(0, 1, 0.05),
    stacked=True,
    color=colors,
)

plt.axvline(t_eff_stats.loc['mean', 't_eff_night_ratio'], color='red', label=f'Mean = {t_eff_stats.loc['mean', 't_eff_night_ratio']:.2f}')
plt.axvline(t_eff_stats.loc['50%', 't_eff_night_ratio'], color='red', linestyle='--', label=f'Median = {t_eff_stats.loc['50%', 't_eff_night_ratio']:.2f}')

plt.xlabel('total effective time / total time in night')
plt.ylabel('# nights')
plt.title('Total effective time / night time colored by moon phase')
plt.tight_layout()
plt.legend()
plt.show()